In [1]:
import pathlib
import sys

import duckdb
import ipywidgets as w
import pandas as pd
from IPython.display import HTML, display

sys.path.insert(0, str(pathlib.Path('../src').resolve()))
import irp.config as _config

cfg = _config.load()
_ROOT = pathlib.Path(_config.__file__).parents[2]
DB = str((_ROOT / cfg['store']['db_path']).resolve())


def q(sql, params=None):
    with duckdb.connect(DB, read_only=True) as con:
        return con.execute(sql, params or []).df()


def _fmt_val(v):
    if pd.isna(v): return ''
    if abs(v) >= 1000: return f'{v:,.0f}'
    if abs(v) >= 1:    return f'{v:,.2f}'
    return f'{v:.4f}'


_STYLE = (
    '<style>'
    ' table.lk { border-collapse: collapse; font-size: 13px; }'
    ' table.lk th { text-align: left; padding: 4px 12px;'
    ' border-bottom: 2px solid #666; white-space: nowrap; }'
    ' table.lk td { text-align: right; padding: 3px 12px; white-space: nowrap; }'
    ' table.lk td:first-child { text-align: left; }'
    ' table.lk tr:last-child td { border-top: 2px solid #666; }'
    '</style>'
)


print('DB:', DB)


DB: /mnt/Dev/active_python_projects/investment_research_platform/data/irp.duckdb


In [2]:
INCOME_ITEMS = [
    'Revenue',
    'Cost of Revenue',
    'Gross Profit',
    'Operating Expenses',
    'Selling, General & Administrative',
    'Research & Development',
    'Depreciation & Amortization',
    'Operating Income (Loss)',
    'Non-Operating Income (Loss)',
    'Interest Expense, Net',
    'Pretax Income (Loss), Adj.',
    'Abnormal Gains (Losses)',
    'Pretax Income (Loss)',
    'Income Tax (Expense) Benefit, Net',
    'Income (Loss) from Continuing Operations',
    'Net Extraordinary Gains (Losses)',
    'Net Income',
    'Net Income (Common)',
    'Shares (Basic)',
    'Shares (Diluted)',
]

BALANCE_ITEMS = [
    'Cash, Cash Equivalents & Short Term Investments',
    'Accounts & Notes Receivable',
    'Inventories',
    'Total Current Assets',
    'Property, Plant & Equipment, Net',
    'Long Term Investments & Receivables',
    'Other Long Term Assets',
    'Total Noncurrent Assets',
    'Total Assets',
    'Payables & Accruals',
    'Short Term Debt',
    'Total Current Liabilities',
    'Long Term Debt',
    'Total Noncurrent Liabilities',
    'Total Liabilities',
    'Share Capital & Additional Paid-In Capital',
    'Treasury Stock',
    'Retained Earnings',
    'Total Equity',
    'Total Liabilities & Equity',
]

CASHFLOW_ITEMS = [
    'Net Income/Starting Line',
    'Depreciation & Amortization',
    'Non-Cash Items',
    'Change in Working Capital',
    'Change in Accounts Receivable',
    'Change in Inventories',
    'Change in Accounts Payable',
    'Change in Other',
    'Net Cash from Operating Activities',
    'Change in Fixed Assets & Intangibles',
    'Net Change in Long Term Investment',
    'Net Cash from Acquisitions & Divestitures',
    'Net Cash from Investing Activities',
    'Dividends Paid',
    'Cash from (Repayment of) Debt',
    'Cash from (Repurchase of) Equity',
    'Net Cash from Financing Activities',
    'Net Change in Cash',
]

_seen = set()
ITEM_ORDER = []
for _item in INCOME_ITEMS + BALANCE_ITEMS + CASHFLOW_ITEMS:
    if _item not in _seen:
        ITEM_ORDER.append(_item)
        _seen.add(_item)

STMT_ITEMS = {
    'All': ITEM_ORDER,
    'Income': INCOME_ITEMS,
    'Balance': BALANCE_ITEMS,
    'Cashflow': CASHFLOW_ITEMS,
}

STMT_TABLES = {
    'All': ['income', 'balance', 'cashflow'],
    'Income': ['income'],
    'Balance': ['balance'],
    'Cashflow': ['cashflow'],
}

_META_COLS = {
    'variant', 'period', 'Ticker', 'Report Date', 'source_id', 'source',
    'Currency', 'Fiscal Year', 'Fiscal Period', 'Publish Date', 'Restated Date', 'inserted_at',
}


def _period_key(p):
    try:
        yr = int(p[:4])
        suf = p[4:]
        return (yr, 5) if suf == 'FY' else (yr, int(suf[1]))
    except Exception:
        return (0, 0)


In [3]:
_all_tickers = sorted(
    q('SELECT DISTINCT Ticker FROM income WHERE Ticker IS NOT NULL')['Ticker'].tolist()
)

ticker_cb = w.Combobox(
    options=_all_tickers,
    placeholder='Ticker…',
    description='Ticker:',
    ensure_option=False,
    layout=w.Layout(width='180px'),
)
stmt_dd = w.Dropdown(
    options=['All', 'Income', 'Balance', 'Cashflow'],
    value='All',
    description='Statement:',
    layout=w.Layout(width='200px'),
)
period_sel = w.SelectMultiple(
    options=[],
    description='Periods:',
    layout=w.Layout(width='160px', height='200px'),
)
item_sel = w.SelectMultiple(
    options=[],
    description='Items:',
    layout=w.Layout(width='420px', height='200px'),
)
show_btn = w.Button(description='Show', button_style='primary', layout=w.Layout(width='100px'))
out = w.Output()


def _update_periods(_=None):
    t = ticker_cb.value.strip().upper()
    period_sel.value = ()
    if not t:
        period_sel.options = []
        return
    periods = set()
    for tbl in ('income', 'balance', 'cashflow'):
        try:
            df = q(f'SELECT DISTINCT period FROM {tbl} WHERE Ticker = ?', [t])
            periods.update(df['period'].dropna().tolist())
        except Exception:
            pass
    opts = sorted(periods, key=_period_key, reverse=True)
    period_sel.options = opts
    period_sel.value = tuple(opts[:8]) if opts else ()


def _update_items(_=None):
    items = STMT_ITEMS[stmt_dd.value]
    item_sel.value = ()
    item_sel.options = items
    item_sel.value = tuple(items)


def _show(_=None):
    out.clear_output(wait=True)
    t = ticker_cb.value.strip().upper()
    periods = list(period_sel.value)
    items = list(item_sel.value)

    if not t or not periods or not items:
        with out:
            print('Select ticker, at least one period, and at least one item.')
        return

    tables = STMT_TABLES[stmt_dd.value]
    frames = []
    ph = ', '.join(['?'] * len(periods))

    for tbl in tables:
        schema = q(f'SELECT * FROM {tbl} LIMIT 0')
        value_cols = [c for c in schema.columns if c not in _META_COLS]
        cols_sql = ', '.join('"' + c + '"' for c in value_cols)
        sql = f'SELECT period, {cols_sql} FROM {tbl} WHERE Ticker = ? AND period IN ({ph})'
        try:
            df = q(sql, [t] + periods)
        except Exception as e:
            with out:
                print(f'Query error ({tbl}): {e}')
            return
        melted = df.melt(id_vars=['period'], var_name='item', value_name='value')
        frames.append(melted)

    if not frames:
        with out:
            print('No data.')
        return

    long = pd.concat(frames, ignore_index=True)
    long = long.drop_duplicates(subset=['period', 'item'], keep='first')
    long = long[long['item'].isin(items)]

    if long.empty:
        with out:
            print('No data for selection.')
        return

    matrix = long.pivot(index='item', columns='period', values='value')
    sorted_cols = sorted(matrix.columns, key=_period_key)
    matrix = matrix[sorted_cols]

    order_map = {item: i for i, item in enumerate(ITEM_ORDER)}
    matrix = matrix.loc[sorted(matrix.index, key=lambda x: order_map.get(x, len(ITEM_ORDER)))]

    display_df = matrix.copy().astype(object)
    for col in display_df.columns:
        display_df[col] = display_df[col].apply(_fmt_val)

    try:
        sf = q(
            'SELECT period, url, form FROM sec_filings WHERE ticker = ? AND period IN (' + ph + ')',
            [t] + periods,
        )
    except Exception:
        sf = pd.DataFrame(columns=['period', 'url', 'form'])

    filing_row = {}
    for col in sorted_cols:
        row = sf[sf['period'] == col]
        if not row.empty:
            u = row.iloc[0]['url']
            fm = row.iloc[0]['form']
            if pd.notna(u) and u:
                filing_row[col] = '<a href="' + u + '" target="_blank">' + (fm or 'filing') + '</a>'
            else:
                filing_row[col] = ''
        else:
            filing_row[col] = ''

    display_df.loc['Filing'] = pd.Series(filing_row)

    with out:
        display(HTML(_STYLE + display_df.to_html(escape=False, classes='lk')))


ticker_cb.observe(_update_periods, names='value')
stmt_dd.observe(_update_items, names='value')
show_btn.on_click(_show)
_update_items()

display(
    w.HBox([
        w.VBox([ticker_cb, stmt_dd, show_btn]),
        period_sel,
        item_sel,
    ]),
    out,
)


Output()